## IMPORTS

In [1]:
import os
import json
import math
import time
import pickle
import numpy as np
import networkx as nx
import osmnx as ox

from geopy.distance import geodesic
from shapely.geometry import Point
from flask import Flask, request, jsonify


### ROAD NETWORK INITIALIZATION

In [2]:
CITY_NAME = "Astana, Kazakhstan"
GRAPH_FILE = "astana_drive.graphml"

ROAD_GRAPH = None


def initialize_road_graph():
    global ROAD_GRAPH

    if ROAD_GRAPH is not None:
        return

    if os.path.exists(GRAPH_FILE):
        ROAD_GRAPH = ox.load_graphml(GRAPH_FILE)
    else:
        ROAD_GRAPH = ox.graph_from_place(CITY_NAME, network_type="drive")
        ox.save_graphml(ROAD_GRAPH, GRAPH_FILE)

    ROAD_GRAPH = ox.add_edge_speeds(ROAD_GRAPH)
    ROAD_GRAPH = ox.add_edge_travel_times(ROAD_GRAPH)


### ROUNTING FUNCTION

In [3]:
def get_drive_time(origin, destination, graph=None):
    G = graph if graph else ROAD_GRAPH

    try:
        origin_node = ox.nearest_nodes(G, origin[1], origin[0])
        dest_node = ox.nearest_nodes(G, destination[1], destination[0])

        path = nx.shortest_path(G, origin_node, dest_node, weight="travel_time")

        total_time = 0
        for i in range(len(path) - 1):
            edge_data = G.get_edge_data(path[i], path[i + 1])
            edge_data = list(edge_data.values())[0]
            total_time += edge_data.get("travel_time", 1)

        return total_time

    except:
        return float("inf")


### PARKING STATE ADAPTER

In [4]:
class ParkingState:

    def __init__(self, start, destination, exit_point, parking_spots, traffic_multiplier=1.4):

        self.start = tuple(start)
        self.destination = tuple(destination)
        self.exit_point = tuple(exit_point)

        self.spots = parking_spots
        self.traffic_multiplier = traffic_multiplier

        # Precompute topology
        self.topology = []
        self.start_progress = None

        self._compute_topology()

    # ---------------------------------------
    # PROJECT SPOTS ON MAIN AVENUE VECTOR
    # ---------------------------------------

    def _compute_topology(self):

        A = self.start
        B = self.exit_point

        dx_ab = B[1] - A[1]
        dy_ab = B[0] - A[0]

        magnitude_ab = math.sqrt(dx_ab**2 + dy_ab**2)

        for spot in self.spots:

            P = spot["coords"]

            dx_ap = P[1] - A[1]
            dy_ap = P[0] - A[0]

            progress = (dx_ap * dx_ab + dy_ap * dy_ab) / max(magnitude_ab, 1e-6)

            cross = (dx_ap * dy_ab) - (dy_ap * dx_ab)
            side = "LEFT" if cross > 0 else "RIGHT"

            spot["progress"] = progress
            spot["side"] = side

            self.topology.append({
                "progress": progress,
                "side": side
            })

        self.start_progress = 0

    # ---------------------------------------
    # DRIVE TIME FUNCTION
    # ---------------------------------------

    def drive_time(self, origin, target):

        if origin == "START":
            origin_coords = self.start
        else:
            origin_coords = origin["coords"]

        if target == "EXIT":
            target_coords = self.exit_point
        else:
            target_coords = target["coords"]

        base_time = get_drive_time(origin_coords, target_coords)

        return base_time * self.traffic_multiplier

    # ---------------------------------------
    # WALK TIME FUNCTION
    # ---------------------------------------

    def walk_time(self, spot):

        distance = geodesic(spot["coords"], self.destination).meters
        return distance / 1.3  # 1.3 m/s walking speed


### expected time calculation

In [5]:
def compute_expected_time(chain, state):

    total_expected_time = 0
    cumulative_failure_probability = 1
    timeline_time = 0

    previous = "START"

    for index in chain:

        spot = state.spots[index]
        p_success = spot.get("p_i", 0.8)
        search_penalty = spot.get("phi_exit_seconds", 60)

        # Drive
        drive = state.drive_time(previous, spot)
        arrival_time = timeline_time + drive

        # Success case
        walk = state.walk_time(spot)
        exit_drive = state.drive_time(spot, "EXIT")

        success_time = arrival_time + walk + exit_drive

        total_expected_time += cumulative_failure_probability * p_success * success_time

        # Failure case
        timeline_time = arrival_time + search_penalty
        cumulative_failure_probability *= (1 - p_success)

        previous = spot

    # All failed penalty
    total_expected_time += cumulative_failure_probability * (timeline_time + 1800)

    confidence = 1 - cumulative_failure_probability

    return total_expected_time, confidence


### OPTIMIZATION ALGORITHM

In [6]:
def optimize_parking_chain(state, k=3):

    number_of_spots = len(state.spots)

    beam = [(0, 1, [], -1)]  # (cost, fail_prob, chain, last_index)

    best_chain = []

    for _ in range(k):

        candidates = []

        for cost_so_far, fail_prob, chain, last_index in beam:

            for i in range(number_of_spots):

                if i in chain:
                    continue

                new_chain = chain + [i]

                expected_time, _ = compute_expected_time(new_chain, state)

                candidates.append((expected_time, fail_prob, new_chain, i))

        candidates.sort(key=lambda x: x[0])
        beam = candidates[:500]

        if beam:
            best_chain = beam[0][2]

    return best_chain


### FLASK SERVER

In [8]:
app = Flask(__name__)
initialize_road_graph()


@app.route("/solve", methods=["POST"])
def solve():

    data = request.json

    start = tuple(data["start"])
    destination = tuple(data["destination"])
    exit_point = tuple(data["exit_point"])

    spots = data["spots"]

    state = ParkingState(start, destination, exit_point, spots)

    best_chain = optimize_parking_chain(state, k=3)

    expected_time, confidence = compute_expected_time(best_chain, state)

    return jsonify({
        "chain": best_chain,
        "expected_time_seconds": expected_time,
        "confidence": confidence
    })


if __name__ == "__main__":
    app.run(debug=False, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [11/Feb/2026 12:46:35] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [11/Feb/2026 12:46:35] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [11/Feb/2026 12:46:40] "GET / HTTP/1.1" 404 -
